# Week 1 - Web Scraping
scraping bbc news articles to build a corpus

In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import time


In [ ]:
BASE_URL = "https://www.bbc.com"
url = "https://www.bbc.com/news"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, "html.parser")

links = []
for a in soup.find_all("a", href=True):
    href = a["href"]
    if href.startswith("/news") and any(i.isdigit() for i in href):
        full_link = BASE_URL + href
        if full_link not in links:
            links.append(full_link)

print(len(links), "links found")


In [ ]:
articles = []

for link in links[:15]:
    try:
        res = requests.get(link, headers=headers)
        s = BeautifulSoup(res.text, "html.parser")

        title = s.find("h1").get_text()
        paragraphs = s.find_all("p")
        text = " ".join([p.get_text() for p in paragraphs])

        if len(text) < 200:
            continue

        articles.append({
            "title": title,
            "text": text
        })
        print("Scraped:", title)
        time.sleep(0.5)

    except Exception as e:
        print("skipped", link)

print("\nTotal:", len(articles))


In [ ]:
with open("corpus.json", "w", encoding="utf-8") as f:
    json.dump(articles, f, indent=4, ensure_ascii=False)


In [ ]:
with open("bbc_news.txt", "w", encoding="utf-8") as f:
    for article in articles:
        f.write(article["title"] + "\n\n")
        f.write(article["text"] + "\n")
        f.write("\n" + "="*50 + "\n\n")

print("Total Articles:", len(articles))
